In [1]:
import pandas as pd 
import re


In [2]:
df = pd.read_csv("../../datasets/raw/mbnz_level1_cleaning.csv")

In [3]:
df.head()

,job_id,title,job_category,start_date,publication,organization,Department,working_mode,location,description,description_en,jd_tasks,jd_qualificaiton
0,MER000441Q,"Manager, Public Relations | Mercedes-Benz > Ca...",Others,01.08.2026,15.06.2026,Mercedes-Benz Malaysia Sdn. Bhd.,SEA Marketing & Digital,Full time,"Mercedes-Benz Malaysia Sdn. Bhd., Kuala Lumpur","TasksOBJECTIVE OF JOB· Develop, plan, a...","TasksOBJECTIVE OF JOB· Develop, plan, a...","OBJECTIVE OF JOB· Develop, plan, and ex...",EDUCATION LEVEL / TRAININGDegree in Communicat...
1,MER0003T59,"Assistant Manager, Tax Specialist | Mercedes-B...",Finance/Controlling,immediately,15.06.2026,Mercedes-Benz Malaysia Sdn. Bhd.,Finance & Controlling MBPLAP,Full time,"Mercedes-Benz Malaysia Sdn. Bhd., Kuala Lumpur",TasksObjective of JobResponsible for managing ...,TasksObjective of JobResponsible for managing ...,Objective of JobResponsible for managing organ...,Education Level / TrainingDegree/Advanced Dipl...
2,MER00044DU,Process Engineer – Automation & Controls | Mer...,Production,immediately,15.06.2026,Mercedes-Benz Manufacturing Poland sp. z o. o.,Engineering,Full time,Mercedes-Benz Manufacturing Poland sp. z o. o....,"TasksLooking for new challenges? The term ""aut...","TasksLooking for new challenges? The term ""aut...","Looking for new challenges? The term ""automati...",- Degree in electrical engineering with...
3,MER00044DN,Full Stack Developer - HUD Miner | Mercedes-Be...,Research and Development incl. Design,01.07.2026,15.06.2026,Mercedes-Benz Research and Development India P...,Production Planning 1,Full time,Mercedes-Benz Research and Development India P...,TasksAbout MBRDI:Mercedes-Benz Research and De...,TasksAbout MBRDI:Mercedes-Benz Research and De...,About MBRDI:Mercedes-Benz Research and Develop...,/ Good to Have:• Mercedes-Benz domain knowledg...
4,MER0003ZHY,C# .NET Developer | Mercedes-Benz > Career > J...,Research and Development incl. Design,immediately,15.06.2026,Mercedes-Benz Research and Development India P...,Central Data Products,Full time,Mercedes-Benz Research and Development India P...,TasksRole: .NET Developer (C#)Experience: +5 y...,TasksRole: .NET Developer (C#)Experience: +5 y...,Role: .NET Developer (C#)Experience: +5 yearsO...,Skills:- 5+ years of hands-on experienc...


In [5]:
df_demo = df.copy()

In [7]:
import re

def clean_text(text):
    try :
        text = text.replace("\xa0", " ")
        text = text.replace("·", " ")
        text = re.sub(r'\s+', ' ', text)   # normalize spaces
        return text.strip()
    except Exception as e : 
        return None

In [8]:
df_demo["jd_qualificaiton"] = df.jd_qualificaiton.apply(clean_text)
df_demo["jd_tasks"] = df.jd_tasks.apply(clean_text)

### Clean Job Description or Qualification

In [41]:
def normalize_unicode(text):
    return text.replace("’", "'").replace("‘", "'").replace("–", "-").replace("—", "-").replace("“", '"').replace("”", '"')
    

In [58]:
import re

def remove_hr_disability_boilerplate(text: str) -> str:
    """
    Removes HR/legal/disability/inclusion boilerplate commonly found
    in job descriptions (especially Mercedes-Benz and similar portals).
    """

    patterns = [

        # -----------------------------
        # 1. Disability inclusion blocks (generic)
        # -----------------------------
        r"(?is)(severely disabled|people with disabilities|disabilities).*?(application process|support you|warmly welcome).*?(\.|$)",

        # -----------------------------
        # 2. HR Services support sentences
        # -----------------------------
        r"(?is)hr services.*?(help|support|questions).*?(\.|$)",

        # -----------------------------
        # 3. SBV contact emails (e.g. sbv-bremen@...)
        # -----------------------------
        r"\b(sbv[-\w]*@\S+)\b",

        # -----------------------------
        # 4. SBV placeholder emails (sbv-[location]@...)
        # -----------------------------
        r"sbv-\[.*?\]@\S+",

        # -----------------------------
        # 5. Role hashtags at end (#Tool mechanic)
        # -----------------------------
        r"#\s*[A-Za-z0-9\s\-\/]+",

        # -----------------------------
        # 6. Mercedes-style repeated inclusion footer
        # -----------------------------
        r"(?is)we particularly welcome applications.*?(process|support you).*?(\.|$)",

        # -----------------------------
        # 7. Generic HR contact closure sentences
        # -----------------------------
        r"(?is)you can also contact.*?(support|process).*?(\.|$)",

    ]

    for p in patterns:
        try : 
            text = re.sub(p, " ", text)
        except Exception as e : 
            print(e)
            
    # cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [121]:
remove_contacts_and_recruitment_sentences(remove_hr_disability_boilerplate(df_demo.jd_qualificaiton.to_list()[216]))

'In the field of communication, media, business or comparableLanguage skills: Very good knowledge of German and English spoken and written, other languages are an advantageIT skills: Confident use of MS Office and ideally experience with image editing and digital mediaPersonal skills: Commitment, ability to work in a team, independent working style as well as strong communication and organizational skillsDesirable: Initial knowledge of Adobe Photoshop, video editing and social mediaAdditional information:We look forward to receiving your online application with CV, cover letter, certificates, current certificate of enrollment with details of the semester and proof of the Standard period of study. com) will be happy to support you in the application process.'

In [57]:
import re
import spacy
nlp = spacy.load("en_core_web_sm")
'''  
part 1) remove contact and time information
part 2) remove application and recruitment sentences
'''
def remove_contacts_and_recruitment_sentences(text: str) -> str:
    """
    Safe version:
    - removes ONLY sentences that are clearly contact/time info
    - avoids deleting skill/requirement sentences accidentally
    """
    doc = nlp(text)
    
    sentences = [sent.text for sent in doc.sents]

    bad_patterns = [
        # part 1
        r'\S+@\S+',                              # emails
        r'\+?\d[\d\s\/\-\(\)]{7,}',             # phones
        r'\d{1,2}\s*(a\.m\.|p\.m\.|am|pm)\s*-\s*\d{1,2}',  # time ranges
        r'\b(mon|tue|wed|thu|fri|sat|sun)\b',    # weekdays (optional)
        r'(?i)\bhr services\b',
        r'(?i)\bhotline\b',
        r'(?i)\breach us\b',
        r'(?i)\bapply exclusively\b' , 
        # part 2 
        r"(?i)\b\d+\s*mb\b",                 # 5 MB, 10 MB, etc.
        r"(?i)relevant to this application",
        r"(?i)recruitment criteria",
        r"(?i)criteria for recruitment",
        r"(?i)selection criteria" , 
        r"(?i)regardless of gender" , 
        r"(?i)setting criteria",


    ]
    bullets = r"[•●▪■·►]"

    cleaned = []

    for idx,s in enumerate(sentences):
        try :
            if any(re.search(p, s) for p in bad_patterns):
                continue
            
            text = re.sub(bullets, "", s)
            text = re.sub("…",". ",text)
            #if idx == 0:
                #text = re.sub(r"^[^:]*:\s*", "", text)
            #text = re.sub(r"(?i)^.*?experience\s*:\s*", ". Experience: ", text) handle the concated words of experiences
            cleaned.append(text)
        except Exception as e: 
            print(e, 
                 type(e).__name__) 
    return " ".join(cleaned)

In [59]:
import re

def clean_skills(text: str) -> str:
    """
    Only cleans skill formatting issues:
    - fixes C,C++ → C, C++
    - fixes spacing in comma-separated skills
    """
    try : 
        text = normalize_unicode(text)    
        # 1. fix missing spaces after commas (C,C++ -> C, C++)
        text = re.sub(r"\s*,\s*", ", ", text)
    
        # 2. fix cases like "C,C++,Python"
        text = re.sub(r",\s*", ", ", text)
    
        # 3. clean double spaces
        text = re.sub(r"\s+", " ", text)
        return text.strip()
    except Exception as e : 
        return text.strip()

### Manage Languages Issues 

In [60]:
from langdetect import detect
from deep_translator import GoogleTranslator

def translate_to_english(text):
    try:
        lang = detect(text)

        if lang == "en":
            return text

        translated = GoogleTranslator(source=lang, target="en").translate(text)
        return translated

    except:
        return text  # fallback if detection fails

In [61]:
import re

def keep_english_only(text: str) -> str:
    """
    Keeps only English part and removes everything from [KOR] onward.
    Also removes [ENG] tag if present.
    """
    try:
        # 1. Cut everything from [KOR] onward
        if "\[KOR\]" in text:
            text = re.sub(r"\[KOR\].*", "", text, flags=re.S)
    
            # 2. Remove [ENG] tag if it exists
            text = re.sub(r"ENG\]|\[ENG\]", " ", text)
    
            # 3. Clean extra spaces
            text = re.sub(r"\s+", " ", text).strip()
       
        return text
    except Exception as e : 
        print(e)
        return text

### apply to job description (qualification part)

In [62]:
def clean_pipeline(job):
    return clean_skills(
        remove_contacts_and_recruitment_sentences(
            remove_hr_disability_boilerplate(
                translate_to_english(
                    keep_english_only(job)
                )
            )
        )
    )

In [64]:
df_demo = df_demo.dropna(subset=["jd_qualificaiton"])

In [65]:
df_demo["cleaned_jd"] = df_demo["jd_qualificaiton"].apply(clean_pipeline)

In [ ]:
df_demo.cleaned_jd.to_list()[50:120]

In [ ]:
for job in df_demo.jd_qualificaiton.to_list()[403:410]: 
    print(
        clean_skills(
            remove_contacts_and_recruitment_sentences(
                remove_hr_disability_boilerplate(
                    translate_to_english(keep_english_only(job)))
        )))

### Clean the `Tasks`

In [70]:
def tasks_cleaner(text): 
    try: 

    
        # Normalize the unicode 
        text = normalize_unicode(text)
        # ==============================
        #   Handle bullet points
        # ======================
        
        bullets = r"[•●▪■·►]"
        text = re.sub(r"(?<!\w)'s\b", "", text)
        text = re.sub(bullets, " ", text)
        text1 = text.split(".")
        filtered = [x for x in text1 if len(x.strip().split()) > 1]      


        
        return ".".join(filtered)
    except Exception as e:
        print(e)

        return text


In [40]:
tasks_cleaner(df_demo.jd_tasks.to_list()[514])

"Objective of jobTo support the implementation of Mercedes-Benz's battery project success, it is critical to restart production and ramp up onsite with planned battery demand within a tight timeline. We are therefore seeking a Production Expert to join our task force at the supplier site, oversee the production ramp-up across module and pack plant, and ensure volume output, quality compliance, and continuous improvement in line with defined KPIs.The Production Expert is assumed as the Technical Project Leader to:Monitoring conformity of the key processes and rework processes, Q-gates, and traceability systems to ensure full adherence to the standards and conduct data analysis to proactively identify potential quality risks.Work with the supplier team to secure output efficiency, product quality and transparency and traceability, set up projects and lead to correct/ improve deviations.In addition, this role will provide transparent updates on production status, product quality, and impr

In [72]:
def clean_tasks_pipeline(job):
    return tasks_cleaner(
        translate_to_english(
            keep_english_only(job)
        )
    )

In [73]:
df_demo["cleaned_tasks"] = df_demo["jd_tasks"].apply(clean_tasks_pipeline)

In [87]:
df_demo.head()

,job_id,title,job_category,start_date,publication,organization,Department,working_mode,location,description_en,cleaned_jd,cleaned_tasks,clean_description
0,MER000441Q,"Manager, Public Relations",Others,01.08.2026,15.06.2026,Mercedes-Benz Malaysia Sdn. Bhd.,SEA Marketing & Digital,Full time,"Mercedes-Benz Malaysia Sdn. Bhd., Kuala Lumpur","TasksOBJECTIVE OF JOB· Develop, plan, a...",EDUCATION LEVEL / TRAININGDegree in Communicat...,"OBJECTIVE OF JOB Develop, plan, and execute MB...","OBJECTIVE OF JOB Develop, plan, and execute MB..."
1,MER0003T59,"Assistant Manager, Tax Specialist",Finance/Controlling,immediately,15.06.2026,Mercedes-Benz Malaysia Sdn. Bhd.,Finance & Controlling MBPLAP,Full time,"Mercedes-Benz Malaysia Sdn. Bhd., Kuala Lumpur",TasksObjective of JobResponsible for managing ...,Education Level / TrainingDegree/Advanced Dipl...,Objective of JobResponsible for managing organ...,Objective of JobResponsible for managing organ...
2,MER00044DU,Process Engineer – Automation & Controls,Production,immediately,15.06.2026,Mercedes-Benz Manufacturing Poland sp. z o. o.,Engineering,Full time,Mercedes-Benz Manufacturing Poland sp. z o. o....,"TasksLooking for new challenges? The term ""aut...",- Degree in electrical engineering with a focu...,"Looking for new challenges? The term ""automati...","Looking for new challenges? The term ""automati..."
3,MER00044DN,Full Stack Developer - HUD Miner,Research and Development incl. Design,01.07.2026,15.06.2026,Mercedes-Benz Research and Development India P...,Production Planning 1,Full time,Mercedes-Benz Research and Development India P...,TasksAbout MBRDI:Mercedes-Benz Research and De...,/ Good to Have: Mercedes-Benz domain knowledge...,About MBRDI:Mercedes-Benz Research and Develop...,About MBRDI:Mercedes-Benz Research and Develop...
4,MER0003ZHY,C# .NET Developer,Research and Development incl. Design,immediately,15.06.2026,Mercedes-Benz Research and Development India P...,Central Data Products,Full time,Mercedes-Benz Research and Development India P...,TasksRole: .NET Developer (C#)Experience: +5 y...,Skills:- 5+ years of hands-on experience in AS...,NET Developer (C#)Experience: +5 yearsOverview...,NET Developer (C#)Experience: +5 yearsOverview...


In [ ]:
df_demo.cleaned_tasks.to_list()[:40]

In [ ]:
for job in df_demo.jd_tasks.to_list()[505:520]:
    print(tasks_cleaner( 
        translate_to_english(
            keep_english_only(job)
        )))

In [75]:
df_demo["clean_description"] =  df_demo.cleaned_tasks + "\n" + df_demo.cleaned_jd

In [76]:
print(df_demo.clean_description.to_list()[0])

OBJECTIVE OF JOB Develop, plan, and execute MB Malaysia's corporate communication & PR strategy to position MBM as a leader in the Malaysian automotive industry and a socially responsible corporate citizen. Drive integrated communications strategies across earned media, search, and AI-driven discovery platforms, leveraging GEO (Generative Engine Optimization) and AEO (Answer Engine Optimization) to strengthen brand visibility, authority, and customer engagement. Champion internal communication strategies to achieve a two-way communication environment that fosters higher morale, greater productivity, and a strong corporate culture. Establish and enhance relationships with external partners including media & influencers, not excluding NGOs and the corporate community, to effectively communicate MB Malaysia's brand presence and image while providing dedicated support. Oversee MB Malaysia's Sponsorship and Donation Database. Oversee and integrate PR and Social Media strategy to enhance bra

In [77]:
df_demo.drop(["jd_tasks","jd_qualificaiton"],axis=1,inplace=True)

In [78]:
df_demo.drop(["description"],axis=1,inplace=True)

In [63]:
df_demo.isna().sum()

job_id               0
title                0
job_category         0
start_date           0
publication          0
organization         0
Department           0
working_mode         0
location             0
description_en       0
clean_description    9
dtype: int64

In [65]:
df_demo = df_demo[~df_demo.clean_description.isna()]

In [88]:
import re
import contractions

def process_text(text):
    
    text = contractions.fix(text)
    # remove emails
    text = re.sub(r'\S+@\S+', '', text)

    # remove urls
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    text = str(text).lower()  # lowercase
    
    text = re.sub(r'\s+', ' ', text)         # remove extra spaces
    text = text.replace("r d", "research and development")
    return text.strip()


In [90]:
df_demo.clean_description= df_demo.clean_description.apply(process_text)

In [91]:
df_demo.clean_description.to_list()[7:10]

["about mbrdi:mercedes-benz research and development india (mbrdi), headquartered in bengaluru with a satellite office in pune, is the largest r&d center for mercedes-benz group ag outside of germany. our mission is to drive innovation and excellence in automotive engineering, digitalization, and sustainable mobility solutions, shaping the future of mobility.key responsibilities:design, develop, and maintain scalable web applications using react, vue.js, java spring boot, and kotlin.build and integrate user-facing features with efficient, reusable components and services.develop and maintain robust restful apis and microservices.work with relational and non-relational databases to manage data operations efficiently.participate in architectural discussions and contribute to the design of scalable, reliable solutions.collaborate with ui/ux designers, product owners, and otheresearch and developmentevelopers in an agile environment.write clean, testable, and efficient code and participate

In [ ]:

clean_titles = [title.split('|', 1)[0].strip() for title in df_demo.title.to_list()]

import re

clean_titles = []

for title in df_demo.title.to_list():
    # Remove Chinese characters
    title = re.sub(r'[\u4e00-\u9fff]+', '', title)

    # Remove everything after first pipe
    title = title.split('|', 1)[0]

    # Normalize spaces
    title = re.sub(r'\s+', ' ', title).strip()
    title = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', title)
    title = re.sub(r'\*(?=[A-Za-zÄÖÜäöüß])', '', title)
    title = translate_to_english(title)


    clean_titles.append(title)

print(clean_titles)

In [92]:
df_demo["title"] = clean_titles

In [93]:
df_demo.head()

,job_id,title,job_category,start_date,publication,organization,Department,working_mode,location,description_en,cleaned_jd,cleaned_tasks,clean_description
0,MER000441Q,"Manager, Public Relations",Others,01.08.2026,15.06.2026,Mercedes-Benz Malaysia Sdn. Bhd.,SEA Marketing & Digital,Full time,"Mercedes-Benz Malaysia Sdn. Bhd., Kuala Lumpur","TasksOBJECTIVE OF JOB· Develop, plan, a...",EDUCATION LEVEL / TRAININGDegree in Communicat...,"OBJECTIVE OF JOB Develop, plan, and execute MB...","objective of job develop, plan, and execute mb..."
1,MER0003T59,"Assistant Manager, Tax Specialist",Finance/Controlling,immediately,15.06.2026,Mercedes-Benz Malaysia Sdn. Bhd.,Finance & Controlling MBPLAP,Full time,"Mercedes-Benz Malaysia Sdn. Bhd., Kuala Lumpur",TasksObjective of JobResponsible for managing ...,Education Level / TrainingDegree/Advanced Dipl...,Objective of JobResponsible for managing organ...,objective of jobresponsible for managing organ...
2,MER00044DU,Process Engineer – Automation & Controls,Production,immediately,15.06.2026,Mercedes-Benz Manufacturing Poland sp. z o. o.,Engineering,Full time,Mercedes-Benz Manufacturing Poland sp. z o. o....,"TasksLooking for new challenges? The term ""aut...",- Degree in electrical engineering with a focu...,"Looking for new challenges? The term ""automati...","looking for new challenges? the term ""automati..."
3,MER00044DN,Full Stack Developer - HUD Miner,Research and Development incl. Design,01.07.2026,15.06.2026,Mercedes-Benz Research and Development India P...,Production Planning 1,Full time,Mercedes-Benz Research and Development India P...,TasksAbout MBRDI:Mercedes-Benz Research and De...,/ Good to Have: Mercedes-Benz domain knowledge...,About MBRDI:Mercedes-Benz Research and Develop...,about mbrdi:mercedes-benz research and develop...
4,MER0003ZHY,C# .NET Developer,Research and Development incl. Design,immediately,15.06.2026,Mercedes-Benz Research and Development India P...,Central Data Products,Full time,Mercedes-Benz Research and Development India P...,TasksRole: .NET Developer (C#)Experience: +5 y...,Skills:- 5+ years of hands-on experience in AS...,NET Developer (C#)Experience: +5 yearsOverview...,net developer (c#)experience: +5 yearsoverview...


In [94]:
df_demo.to_csv("../../datasets/processed/mdz_cleaned.v2.csv",index=False)